In [1]:
import dspy
llama32 = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='')
gpt_oss = dspy.LM('ollama_chat/gpt-oss:latest', api_base='http://localhost:11434', api_key='')

# Modules

## How do I use a built-in module, like dspy.Predict or dspy.ChainOfThought?

In [22]:
with dspy.context(lm=llama32):
    sentence = "it's a charming and often affecting journey."  # example from the SST-2 dataset.

    # 1) Declare with a signature.
    classify = dspy.Predict('sentence -> sentiment: bool')

    # 2) Call with input argument(s). 
    response = classify(sentence=sentence)

    # 3) Access the output.
    print(response.sentiment)

True


In [28]:
with dspy.context(lm=llama32):
    question = "What's something great about the ColBERT retrieval model?"

    # 1) Declare with a signature, and pass some config.
    classify = dspy.ChainOfThought('question -> answer', n=5, allowed_openai_params=['n'])

    # 2) Call with input argument.
    response = classify(question=question)

    # 3) Access the outputs.
    print(response.completions.answer)
    print(f"Reasoning: {response.reasoning}")
    print(f"Answer: {response.answer}")

['Its ability to effectively rank documents based on their relevance to a query.']
Reasoning: The ColBERT retrieval model is known for its high performance in ranking low-resource languages, making it a great tool for language translation and information retrieval tasks.
Answer: Its ability to effectively rank documents based on their relevance to a query.


## What other DSPy modules are there? How can I use them?

[Build DSPy modules for various tasks.ipynb](../Get%20Started/Build%20DSPy%20modules%20for%20various%20tasks.ipynb)

## How do I compose multiple modules into a bigger program?

[Tutorial: Multi-Hop Retrieval](https://dspy.ai/tutorials/multihop_search/)

## How do I track LM usage?

In [30]:
import dspy

# Define a simple program that makes multiple LM calls
class MyProgram(dspy.Module):
    def __init__(self):
        self.predict1 = dspy.ChainOfThought("question -> answer")
        self.predict2 = dspy.ChainOfThought("question, answer -> score")

    def __call__(self, question: str) -> str:
        answer = self.predict1(question=question)
        score = self.predict2(question=question, answer=answer)
        return score

# Configure DSPy with tracking enabled
with dspy.context(
    lm=llama32,
    track_usage=True
):
    # Run the program and check usage
    program = MyProgram()
    output = program(question="What is the capital of France?")
    print(output.get_lm_usage())

{'ollama_chat/llama3.2': {'completion_tokens': 36, 'prompt_tokens': 239, 'total_tokens': 275, 'completion_tokens_details': None, 'prompt_tokens_details': None}}
